# Kaggle Training — RawNet2-Mini with Raw Waveform
## ASVspoof 2019 Logical Access

**EDA-informed hyperparameters:**
- Raw waveform `[1, 64000]` — bypasses handcrafted features entirely
- SincNet learns 128 bandpass filters with learnable cutoff frequencies
- EDA (nb 05): phase-level artifacts are waveform-domain phenomena — raw model captures these
- Lower LR=1e-4 (SincNet filter init is sensitive)
- FMS (Feature Map Scaling) for learned channel attention on sinc-filtered features


In [ ]:
import os, glob

POSSIBLE_ROOTS = [
    "/kaggle/input/asvpoof-2019-dataset",
    "/kaggle/input/asvpoof2019",
    "/kaggle/input/asvspoof-2019",
    "/kaggle/input/asvpoof-2019",
    "/kaggle/input/la-asvspoof2019",
]
DATA_ROOT = None
for p in POSSIBLE_ROOTS:
    if os.path.exists(p):
        DATA_ROOT = p
        break
if DATA_ROOT is None:
    matches = glob.glob("/kaggle/input/**/ASVspoof2019_LA_train", recursive=True)
    if matches:
        DATA_ROOT = matches[0].replace("/ASVspoof2019_LA_train", "")
if DATA_ROOT is None:
    raise RuntimeError("Dataset not found. Add ASVspoof 2019 LA dataset to this notebook.")
print(f"Dataset root: {DATA_ROOT}")


In [ ]:
import subprocess
subprocess.run(["pip", "install", "soundfile", "librosa", "-q"])


In [ ]:
import torch

CFG = {
    "sample_rate":    16000,
    "target_samples": 64000,
    "pre_emphasis":   0.97,
    "vad_top_db":     40,

    "sinc_out_channels": 128,   # Number of learned bandpass filters
    "sinc_kernel_size":  251,   # Filter length (odd) — ~15ms at 16kHz
    "sinc_stride":       1,

    "batch_size":     32,       # Smaller: raw 64k samples per item uses more memory
    "epochs":         30,
    "lr":             1e-4,     # Lower LR: SincNet filter learning is sensitive
    "lr_min":         1e-6,
    "weight_decay":   1e-4,
    "focal_alpha":    0.75,
    "focal_gamma":    2.0,
    "label_smoothing": 0.05,
    "dropout":        0.3,

    "seed":           42,
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "num_workers":    2,
    "output_dir":     "/kaggle/working",
    "model_name":     "rawnet_mini",
}

torch.manual_seed(CFG["seed"])
import numpy as np; np.random.seed(CFG["seed"])
print(f"Device: {CFG['device']}")
if CFG["device"] == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
import pandas as pd

def parse_protocols(data_root):
    split_map = {"train": "ASVspoof2019_LA_train", "dev": "ASVspoof2019_LA_dev", "eval": "ASVspoof2019_LA_eval"}
    rows = []
    for split, flac_subdir in split_map.items():
        flac_dir = os.path.join(data_root, flac_subdir, "flac")
        proto_files = glob.glob(os.path.join(data_root, "**", f"*{split[:3]}*.txt"), recursive=True)
        if not proto_files:
            print(f"WARNING: No protocol for {split}"); continue
        with open(proto_files[0]) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                spk, aid, _, atk, key = parts[0], parts[1], parts[2], parts[3], parts[4]
                fp = os.path.join(flac_dir, aid + ".flac")
                rows.append({"audio_id": aid, "attack_id": atk, "key": key,
                             "is_spoof": 1 if key == "spoof" else 0,
                             "split": split, "file_path": fp, "file_exists": os.path.exists(fp)})
    return pd.DataFrame(rows)

manifest = parse_protocols(DATA_ROOT)
for split in ["train", "dev"]:
    sub = manifest[manifest.split == split]
    bon = len(sub[sub.key == "bonafide"])
    spf = len(sub[sub.key == "spoof"])
    print(f"{split}: total={len(sub)} bon={bon} spoof={spf} missing={len(sub[~sub.file_exists])}")


In [ ]:
import numpy as np, soundfile as sf, librosa
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

def load_waveform(filepath, is_training=False, cfg=CFG):
    y, sr = sf.read(filepath)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if sr != cfg["sample_rate"]:
        y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=cfg["sample_rate"])
    y = y.astype(np.float32)
    y = np.concatenate([[y[0]], y[1:] - cfg["pre_emphasis"] * y[:-1]])
    intervals = librosa.effects.split(y=y, top_db=cfg["vad_top_db"])
    if len(intervals) > 0:
        trimmed = np.concatenate([y[s:e] for s, e in intervals])
        if len(trimmed) > 1000:
            y = trimmed
    T = cfg["target_samples"]
    n = len(y)
    if n >= T:
        start = np.random.randint(0, n - T + 1) if is_training else (n - T) // 2
        y = y[start:start + T]
    else:
        y = np.pad(y, (0, T - n), mode="wrap")
    return y / (np.max(np.abs(y)) + 1e-7)

class RawWaveformDataset(Dataset):
    def __init__(self, df, is_training=False, cfg=CFG):
        self.df = df.reset_index(drop=True)
        self.is_training = is_training
        self.cfg = cfg

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            y = load_waveform(row.file_path, self.is_training, self.cfg)
        except Exception:
            y = np.zeros(self.cfg["target_samples"], dtype=np.float32)
        return torch.from_numpy(y).unsqueeze(0), torch.tensor(int(row.is_spoof), dtype=torch.long)

train_df = manifest[manifest.split == "train"].reset_index(drop=True)
dev_df   = manifest[manifest.split == "dev"].reset_index(drop=True)

labels = train_df.is_spoof.values
class_counts = np.bincount(labels)
sample_weights = torch.FloatTensor((1.0 / class_counts)[labels])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = RawWaveformDataset(train_df, is_training=True)
dev_ds   = RawWaveformDataset(dev_df, is_training=False)
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,
                          num_workers=CFG["num_workers"], pin_memory=True)
dev_loader   = DataLoader(dev_ds, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
print(f"Train: {len(train_ds)} | Dev: {len(dev_ds)} | Batches/epoch: {len(train_loader)}")

# Verify batch
xb, yb = next(iter(train_loader))
print(f"Batch shapes: x={xb.shape}, y={yb.shape}")


In [ ]:
import torch.nn as nn, torch.nn.functional as F, math

class SincConv(nn.Module):
    def __init__(self, out_channels, kernel_size, sample_rate=16000, in_channels=1):
        super().__init__()
        assert kernel_size % 2 == 1
        assert in_channels == 1
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate

        # Init filter bank: uniformly spaced from 50Hz to Nyquist
        low_hz = 50.0
        high_hz = sample_rate / 2 - 50.0
        mel_low = 2595 * math.log10(1 + low_hz / 700)
        mel_high = 2595 * math.log10(1 + high_hz / 700)
        mel_pts = torch.linspace(mel_low, mel_high, out_channels + 2)
        hz_pts = 700 * (10 ** (mel_pts / 2595) - 1)

        self.low_hz_ = nn.Parameter(hz_pts[:-2].unsqueeze(1))
        self.band_hz_ = nn.Parameter((hz_pts[1:-1] - hz_pts[:-2]).unsqueeze(1))

        n = (kernel_size - 1) / 2.0
        self.register_buffer("n_", 2 * math.pi * torch.arange(-n, 0).view(1, -1) / sample_rate)
        self.register_buffer("window_", torch.hamming_window(kernel_size)[:(kernel_size//2)])

    def forward(self, x):
        low = torch.clamp(self.low_hz_, min=50.0) / self.sample_rate
        high = torch.clamp(low + torch.clamp(self.band_hz_, min=50.0) / self.sample_rate, max=0.499)
        f_times_t_low = torch.matmul(low, self.n_)
        f_times_t_high = torch.matmul(high, self.n_)
        band_pass_left = (torch.sin(f_times_t_high) - torch.sin(f_times_t_low)) / (self.n_ / 2) * self.window_
        band_pass_center = 2 * (high - low)
        band_pass_right = torch.flip(band_pass_left, dims=[1])
        band_pass = torch.cat([band_pass_left, band_pass_center, band_pass_right], dim=1)
        band_pass = band_pass / (2 * band_pass.norm(dim=1, keepdim=True) + 1e-8)
        return F.conv1d(x, band_pass.unsqueeze(1), stride=CFG["sinc_stride"], padding=self.kernel_size//2)

class FMSBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(channels, 1))
        self.shift = nn.Parameter(torch.zeros(channels, 1))

    def forward(self, x):
        s = torch.sigmoid(x.mean(dim=2, keepdim=True))
        return x * (self.scale * s + self.shift)

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm1d(out_ch), nn.LeakyReLU(0.3),
            nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        self.fms = FMSBlock(out_ch)
        self.shortcut = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if stride != 1 or in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return F.leaky_relu(self.fms(self.conv(x)) + self.shortcut(x), 0.3)

class AttentiveStatsPool(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(nn.Conv1d(channels, 128, 1), nn.Tanh(), nn.Conv1d(128, channels, 1), nn.Softmax(dim=2))

    def forward(self, x):
        w = self.attention(x)
        mu = (w * x).sum(dim=2)
        sigma = (w * (x - mu.unsqueeze(2)) ** 2).sum(dim=2).sqrt()
        return torch.cat([mu, sigma], dim=1)

class RawNetMini(nn.Module):
    def __init__(self, cfg=CFG):
        super().__init__()
        self.sinc = SincConv(cfg["sinc_out_channels"], cfg["sinc_kernel_size"])
        self.bn0 = nn.BatchNorm1d(cfg["sinc_out_channels"])
        self.blocks = nn.Sequential(
            ResBlock1D(128, 128, stride=3),
            ResBlock1D(128, 128, stride=3),
            ResBlock1D(128, 256, stride=3),
            ResBlock1D(256, 256, stride=3),
            ResBlock1D(256, 256, stride=3),
            ResBlock1D(256, 256, stride=3),
        )
        self.pool = AttentiveStatsPool(256)
        self.head = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(),
            nn.Dropout(cfg["dropout"]),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = torch.abs(self.sinc(x))
        x = F.leaky_relu(self.bn0(x), 0.3)
        x = F.max_pool1d(x, 3)
        x = self.blocks(x)
        x = self.pool(x)
        return self.head(x)

device = torch.device(CFG["device"])
model = RawNetMini(CFG).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"RawNet-Mini parameters: {n_params:,}")
with torch.no_grad():
    dummy = torch.zeros(2, 1, 64000).to(device)
    out = model(dummy)
    print(f"Output shape: {out.shape}  (expected [2, 2])")


In [ ]:
import time, json
from sklearn.metrics import roc_auc_score

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha, self.gamma, self.label_smoothing = alpha, gamma, label_smoothing
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction="none", label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        alpha_t = torch.where(targets == 1, self.alpha, 1.0 - self.alpha)
        return (alpha_t * (1 - pt) ** self.gamma * ce).mean()

def compute_eer(y_true, y_score):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return float((fpr[idx] + fnr[idx]) / 2)

criterion = FocalLoss(CFG["focal_alpha"], CFG["focal_gamma"], CFG["label_smoothing"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=CFG["lr_min"])
scaler = torch.cuda.amp.GradScaler(enabled=(CFG["device"] == "cuda"))

best_eer, best_auc = float("inf"), 0.0
history = []
save_path = os.path.join(CFG["output_dir"], f"{CFG['model_name']}_best.pth")

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss, t0 = 0.0, time.time()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * xb.size(0)
    train_loss = total_loss / len(train_ds)
    scheduler.step(epoch)

    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for xb, yb in dev_loader:
            with torch.cuda.amp.autocast(enabled=(CFG["device"] == "cuda")):
                logits = model(xb.to(device))
            all_probs.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            all_targets.append(yb.numpy())
    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    eer = compute_eer(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    acc = ((y_prob >= 0.5) == y_true).mean()

    print(f"Ep {epoch:02d}/{CFG['epochs']} | loss={train_loss:.4f} | EER={eer*100:.2f}% | AUC={auc:.4f} | acc={acc*100:.1f}% | {time.time()-t0:.0f}s")
    history.append({"epoch": epoch, "train_loss": round(train_loss,6), "val_eer": round(eer,6), "val_auc": round(auc,6)})

    if eer < best_eer:
        best_eer, best_auc = eer, auc
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "eer": best_eer, "auc": best_auc, "cfg": CFG}, save_path)
        print(f"  >>> Best: EER={best_eer*100:.2f}% AUC={best_auc:.4f}")

json.dump(history, open(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_history.json"), "w"), indent=2)
print(f"\nBest EER: {best_eer*100:.2f}% | Best AUC: {best_auc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_list, [h["train_loss"] for h in history], color="#3498db", lw=2)
axes[0].set_title("Training Loss"); axes[0].set_xlabel("Epoch")
eers = [h["val_eer"]*100 for h in history]
axes[1].plot(epochs_list, eers, color="#e74c3c", lw=2, marker="o", ms=4)
axes[1].axhline(min(eers), color="gray", ls="--", label=f"Best: {min(eers):.2f}%"); axes[1].legend()
axes[1].set_title("Dev EER (%)")
aucs = [h["val_auc"] for h in history]
axes[2].plot(epochs_list, aucs, color="#2ecc71", lw=2, marker="o", ms=4)
axes[2].axhline(max(aucs), color="gray", ls="--", label=f"Best: {max(aucs):.4f}"); axes[2].legend()
axes[2].set_title("Dev AUC")
plt.tight_layout()
plt.savefig(os.path.join(CFG["output_dir"], f"{CFG['model_name']}_curve.png"), dpi=150)
plt.show()
print(f"Best EER: {best_eer*100:.2f}% | AUC: {best_auc:.4f}")
